### Classification without RUS and SMOTE - (SVM, TSF, and GRU)

In [2]:
# ══════════════════════════════════════════════════════════════
# BLOCK 1 — Load All Three Normalized Datasets
# Shape: (samples, 288 timepoints, 10 channels)
# Format: dict with keys 'X' and 'y'
# ══════════════════════════════════════════════════════════════
import pickle
import numpy as np

DATA_PATHS = {
    "per_file": {
        "train": "./final_split_data_FileNorm/train_set.pkl",
        "test":  "./final_split_data_FileNorm/test_set.pkl",
    },
    "per_row": {
        "train": "./final_split_data_RowNorm/train_set.pkl",
        "test":  "./final_split_data_RowNorm/test_set.pkl",
    },
    "no_norm": {
        "train": "./final_split_data_noNorm/train_set.pkl",
        "test":  "./final_split_data_noNorm/test_set.pkl",
    },
}

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

datasets = {}

for norm_name, splits in DATA_PATHS.items():
    X_train, y_train = load_split(splits["train"])
    X_test,  y_test  = load_split(splits["test"])
    datasets[norm_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_test":  X_test,  "y_test":  y_test,
    }

# ── Summary ────────────────────────────────────────────────────
print(f"{'Norm':<10} {'Train shape':<22} {'Test shape':<22} {'Class 0':>8} {'Class 1':>8} {'Imbalance ratio':>16}")
print("─" * 90)
for norm_name, d in datasets.items():
    counts = np.bincount(d["y_train"])
    ratio  = counts[0] / counts[1]
    print(f"{norm_name:<10} {str(d['X_train'].shape):<22} {str(d['X_test'].shape):<22} "
          f"{counts[0]:>8} {counts[1]:>8} {ratio:>15.1f}x")

print(f"\n⚠️  Severe class imbalance detected (~{ratio:.0f}:1).")
print("   Consider using class_weight='balanced' in SVM,")
print("   and class_weight={{0:1, 1:{:.0f}}} in GRU.".format(ratio))

Norm       Train shape            Test shape              Class 0  Class 1  Imbalance ratio
──────────────────────────────────────────────────────────────────────────────────────────
per_file   (12455, 288, 10)       (3559, 288, 10)           12337      118           104.6x
per_row    (12455, 288, 10)       (3559, 288, 10)           12337      118           104.6x
no_norm    (12455, 288, 10)       (3559, 288, 10)           12337      118           104.6x

⚠️  Severe class imbalance detected (~105:1).
   Consider using class_weight='balanced' in SVM,
   and class_weight={0:1, 1:105} in GRU.


### Vectorization for SVM

In [3]:
# ══════════════════════════════════════════════════════════════
# BLOCK 1b — Catch22 Feature Extraction for SVM
#
# Pipeline:
#   (N, 288, 10)  ──transpose──►  (N, 10, 288)   [sktime panel format]
#                 ──Catch22──────►  (N, 10, 22)   [22 features / channel]
#                 ──flatten──────►  (N, 220)       [SVM-ready 2-D input]
#
# Saves each normalisation's train/test arrays to:
#   ./catch22_features/<norm_name>_train.npz
#   ./catch22_features/<norm_name>_test.npz
# ══════════════════════════════════════════════════════════════
import os
import numpy as np
from joblib import Parallel, delayed
from sktime.transformations.panel.catch22 import Catch22

SAVE_DIR = "./catch22_features"
os.makedirs(SAVE_DIR, exist_ok=True)

def _catch22_chunk(X_chunk: np.ndarray) -> np.ndarray:
    """Run Catch22 on a subset of samples and return numpy array."""
    transformer = Catch22(catch24=False)
    return transformer.fit_transform(X_chunk).to_numpy()

def extract_catch22(X: np.ndarray, label: str = "", n_jobs: int = -1, chunk_size: int = 500) -> np.ndarray:
    """
    Parameters
    ----------
    X          : (n_samples, n_timepoints, n_channels)
    label      : optional label for progress printing
    n_jobs     : joblib parallel workers (-1 = all cores)
    chunk_size : samples per parallel chunk
    Returns
    -------
    np.ndarray, shape (n_samples, 22 * n_channels)
    """
    n_samples, n_timepoints, n_channels = X.shape

    # ── guard against NaN / Inf ────────────────────────────────
    if not np.isfinite(X).all():
        n_bad = (~np.isfinite(X)).sum()
        print(f"    ⚠️  {n_bad} non-finite values in {label} — replacing with per-channel median")
        X = X.copy()
        for c in range(n_channels):
            col = X[:, :, c]
            col[~np.isfinite(col)] = np.nanmedian(col)
            X[:, :, c] = col

    # ── transpose to sktime format ─────────────────────────────
    X_sktime = X.transpose(0, 2, 1).astype(np.float64)   # (N, 10, 288)

    # ── split into chunks for parallel processing ──────────────
    chunks = [X_sktime[i:i + chunk_size] for i in range(0, n_samples, chunk_size)]
    print(f"    Processing {n_samples} samples in {len(chunks)} chunks on {n_jobs} workers …", flush=True)

    results = Parallel(n_jobs=n_jobs)(
        delayed(_catch22_chunk)(chunk) for chunk in chunks
    )

    X_feat = np.vstack(results)   # (N, 220)

    expected_cols = 22 * n_channels
    assert X_feat.shape == (n_samples, expected_cols), (
        f"Unexpected catch22 output shape: {X_feat.shape}  "
        f"(expected ({n_samples}, {expected_cols}))"
    )
    return X_feat

# ── Sanity check raw value ranges before extraction ────────────
print("Pre-extraction value range check:")
print(f"{'Norm':<10} {'Finite?':<10} {'Min':>15} {'Max':>15} {'NaN count':>12}")
print("─" * 65)
for norm_name, d in datasets.items():
    X = d["X_train"]
    print(f"{norm_name:<10} {str(np.isfinite(X).all()):<10} {X.min():>15.4f} {X.max():>15.4f} {(~np.isfinite(X)).sum():>12}")

# ── Extraction + save loop ─────────────────────────────────────
print("\nExtracting & saving catch22 features …")
print(f"{'Norm':<10} {'Train shape':<22} {'Test shape':<22} {'Saved to'}")
print("─" * 80)

datasets_svm = {}
SKIP_FOR_SVM = {"no_norm"}

for norm_name, d in datasets.items():
    if norm_name in SKIP_FOR_SVM:
        print(f"\n  [{norm_name}] skipped for catch22/SVM — will still run ROCKET + GRU in Block 2")
        continue
    print(f"\n  [{norm_name}] extracting train …", flush=True)
    X_tr = extract_catch22(d["X_train"], label=f"{norm_name}/train")

    print(f"  [{norm_name}] extracting test  …", flush=True)
    X_te = extract_catch22(d["X_test"],  label=f"{norm_name}/test")

    # ── save ──────────────────────────────────────────────────
    np.savez(os.path.join(SAVE_DIR, f"{norm_name}_train.npz"), X=X_tr, y=d["y_train"])
    np.savez(os.path.join(SAVE_DIR, f"{norm_name}_test.npz"),  X=X_te, y=d["y_test"])

    # ── keep in-memory mirror for same-session use ─────────────
    datasets_svm[norm_name] = {
        "X_train": X_tr, "y_train": d["y_train"],
        "X_test":  X_te, "y_test":  d["y_test"],
    }

    print(f"  [{norm_name}] ✓  train {X_tr.shape}  test {X_te.shape}  → {SAVE_DIR}/")

print(f"\n✅  Catch22 extraction complete.")
print(f"    Folder : {os.path.abspath(SAVE_DIR)}/")
print(f"    Files  : <norm>_train.npz  and  <norm>_test.npz  (keys: 'X', 'y')")
print(f"    Shape  : 22 features × 10 channels = 220 cols per sample")

Pre-extraction value range check:
Norm       Finite?                Min             Max    NaN count
─────────────────────────────────────────────────────────────────
per_file   True               -0.2459        136.5618            0
per_row    True               -0.5445         32.3689            0
no_norm    True            -1126.2000    6485041.0000            0

Extracting & saving catch22 features …
Norm       Train shape            Test shape             Saved to
────────────────────────────────────────────────────────────────────────────────

  [per_file] extracting train …
    Processing 12455 samples in 25 chunks on -1 workers …
  [per_file] extracting test  …
    Processing 3559 samples in 8 chunks on -1 workers …
  [per_file] ✓  train (12455, 220)  test (3559, 220)  → ./catch22_features/

  [per_row] extracting train …
    Processing 12455 samples in 25 chunks on -1 workers …
  [per_row] extracting test  …
    Processing 3559 samples in 8 chunks on -1 workers …
  [per_row] ✓

### Missing value imputation from Catch22

In [10]:
from sklearn.impute import SimpleImputer

SAVE_DIR = "./catch22_features"

datasets_svm = {}
for fname in os.listdir(SAVE_DIR):
    if not fname.endswith("_train.npz"):
        continue
    norm_name = fname.replace("_train.npz", "")
    tr = np.load(os.path.join(SAVE_DIR, f"{norm_name}_train.npz"))
    te = np.load(os.path.join(SAVE_DIR, f"{norm_name}_test.npz"))

    # ── pull everything out of the file handles immediately ────
    X_tr, y_tr = tr["X"], tr["y"]
    X_te, y_te = te["X"], te["y"]

    # ── impute NaNs produced by catch22 ────────────────────────
    n_nan_tr = np.isnan(X_tr).sum()
    n_nan_te = np.isnan(X_te).sum()
    if n_nan_tr > 0 or n_nan_te > 0:
        print(f"    ⚠️  [{norm_name}] NaNs in catch22 output — "
              f"train: {n_nan_tr}  test: {n_nan_te}  → imputing with column median")
        imp = SimpleImputer(strategy="median")
        X_tr = imp.fit_transform(X_tr)
        X_te = imp.transform(X_te)

        # ── overwrite saved files with clean versions ──────────
        np.savez(os.path.join(SAVE_DIR, f"{norm_name}_train.npz"), X=X_tr, y=y_tr)
        np.savez(os.path.join(SAVE_DIR, f"{norm_name}_test.npz"),  X=X_te, y=y_te)
        print(f"    ✓  [{norm_name}] cleaned files saved to {SAVE_DIR}/")
    else:
        print(f"    ✓  [{norm_name}] no NaNs found — files unchanged")

    datasets_svm[norm_name] = {
        "X_train": X_tr, "y_train": y_tr,
        "X_test":  X_te, "y_test":  y_te,
    }

    ✓  [per_file] no NaNs found — files unchanged
    ⚠️  [per_row] NaNs in catch22 output — train: 518046  test: 146231  → imputing with column median
    ✓  [per_row] cleaned files saved to ./catch22_features/


### Running SVM, ROCKET, and GRU on the data

In [11]:
# ══════════════════════════════════════════════════════════════
# BLOCK 2 — SVM / ROCKET / GRU per Normalization
# Loads all data from disk — safe to run after kernel restart.
#
# Requires:
#   ./catch22_features/<norm>_train.npz   (from Block 1b)
#   ./catch22_features/<norm>_test.npz
#   ./final_split_data_FileNorm/train_set.pkl  etc. (from Block 1)
# ══════════════════════════════════════════════════════════════
import os
import pickle
import numpy as np
import pandas as pd
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, f1_score,
                             classification_report, confusion_matrix)
from sktime.classification.kernel_based import RocketClassifier
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.utils import to_categorical
import warnings
warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════
# LOAD RAW DATASETS  (3-D arrays for ROCKET + GRU)
# ══════════════════════════════════════════════════════════════
DATA_PATHS = {
    "per_file": {
        "train": "./final_split_data_FileNorm/train_set.pkl",
        "test":  "./final_split_data_FileNorm/test_set.pkl",
    },
    "per_row": {
        "train": "./final_split_data_RowNorm/train_set.pkl",
        "test":  "./final_split_data_RowNorm/test_set.pkl",
    },
    # "no_norm": {
    #     "train": "./final_split_data_noNorm/train_set.pkl",
    #     "test":  "./final_split_data_noNorm/test_set.pkl",
    # },
}

def load_split(path):
    with open(path, "rb") as f:
        data = pickle.load(f)
    return data["X"].astype(np.float32), data["y"]

datasets = {}
for norm_name, splits in DATA_PATHS.items():
    X_train, y_train = load_split(splits["train"])
    X_test,  y_test  = load_split(splits["test"])
    datasets[norm_name] = {
        "X_train": X_train, "y_train": y_train,
        "X_test":  X_test,  "y_test":  y_test,
    }

print("✅  Raw datasets loaded:")
for norm_name, d in datasets.items():
    print(f"    {norm_name:<10}  train {d['X_train'].shape}  test {d['X_test'].shape}")

# ══════════════════════════════════════════════════════════════
# LOAD CATCH22 DATASETS  (flat 2-D arrays for SVM)
# ══════════════════════════════════════════════════════════════
SAVE_DIR = "./catch22_features"

datasets_svm = {}
for fname in os.listdir(SAVE_DIR):
    if not fname.endswith("_train.npz"):
        continue
    norm_name = fname.replace("_train.npz", "")
    tr = np.load(os.path.join(SAVE_DIR, f"{norm_name}_train.npz"))
    te = np.load(os.path.join(SAVE_DIR, f"{norm_name}_test.npz"))
    datasets_svm[norm_name] = {
        "X_train": tr["X"], "y_train": tr["y"],
        "X_test":  te["X"], "y_test":  te["y"],
    }

print("\n✅  Catch22 datasets loaded:")
for norm_name, d in datasets_svm.items():
    print(f"    {norm_name:<10}  train {d['X_train'].shape}  test {d['X_test'].shape}")

# ── Confirm keys match between the two dicts ──────────────────
assert set(datasets.keys()) == set(datasets_svm.keys()), (
    f"Key mismatch!\n  datasets:     {set(datasets.keys())}\n"
    f"  datasets_svm: {set(datasets_svm.keys())}"
)
print("\n✅  Key alignment confirmed:", sorted(datasets.keys()))

# ══════════════════════════════════════════════════════════════
# CLASS WEIGHTS  (12337 vs 118 from Block 1)
# ══════════════════════════════════════════════════════════════
CLASS_WEIGHT_RATIO = 12337 / 118   # ≈ 104.5
CLASS_WEIGHTS      = {0: 1.0, 1: CLASS_WEIGHT_RATIO}

# ── Reshape helpers ────────────────────────────────────────────
def to_svm(X):
    """(s, 220) — already flat from catch22, pass through"""
    return X

def to_rocket(X):
    """(s, 288, 10) → (s, 10, 288)  [sktime: samples, channels, timepoints]"""
    return X.transpose(0, 2, 1)

def to_gru(X):
    """(s, 288, 10) → already correct for Keras GRU"""
    return X

# ── TSS metric ─────────────────────────────────────────────────
def tss_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    recall      = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    false_alarm = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    return recall - false_alarm

# ── Metric helpers ─────────────────────────────────────────────
def compute_metrics(y_true, y_pred):
    report = classification_report(y_true, y_pred,
                                   target_names=["Class 0", "Class 1"],
                                   output_dict=True)
    return {
        "accuracy":         round(accuracy_score(y_true, y_pred), 4),
        "macro_f1":         round(f1_score(y_true, y_pred, average="macro"), 4),
        "class1_f1":        round(report["Class 1"]["f1-score"],  4),
        "class1_recall":    round(report["Class 1"]["recall"],    4),
        "class1_precision": round(report["Class 1"]["precision"], 4),
        "tss":              round(tss_score(y_true, y_pred),      4),
        "cm":               confusion_matrix(y_true, y_pred),
        "report":           report,
    }

def print_metrics(metrics, clf_name):
    print(f"\n  ── [{clf_name}] ──────────────────────────────────────")
    print(f"  TSS:            {metrics['tss']:.4f}  ← primary metric")
    print(f"  Macro F1:       {metrics['macro_f1']:.4f}")
    print(f"  Class 1 F1:     {metrics['class1_f1']:.4f}")
    print(f"  Class 1 Recall: {metrics['class1_recall']:.4f}")
    print(f"  Class 1 Prec.:  {metrics['class1_precision']:.4f}")
    print(f"  Accuracy:       {metrics['accuracy']:.4f}  (misleading at 104:1)")
    tn, fp, fn, tp = metrics['cm'].ravel()
    print(f"  Confusion Matrix → TP:{tp}  FP:{fp}  TN:{tn}  FN:{fn}")

# ══════════════════════════════════════════════════════════════
# MAIN LOOP — SVM / ROCKET / GRU per normalization
# ══════════════════════════════════════════════════════════════
results = {}

for norm_name, d in datasets.items():

    X_tr_raw, y_tr = d["X_train"], d["y_train"]
    X_te_raw, y_te = d["X_test"],  d["y_test"]
    X_tr_svm       = datasets_svm[norm_name]["X_train"]
    X_te_svm       = datasets_svm[norm_name]["X_test"]

    n_samples, n_timepoints, n_channels = X_tr_raw.shape
    n_classes = 2

    print(f"\n{'═'*60}")
    print(f"  NORMALIZATION: {norm_name.upper()}")
    print(f"{'═'*60}")

    norm_results = {}

    # ── [A] SVM ───────────────────────────────────────────────────
    if norm_name in datasets_svm:
        svm = SVC(kernel="rbf", C=1.0, class_weight="balanced", random_state=42)
        svm.fit(to_svm(X_tr_svm), y_tr)
        norm_results["SVM"] = compute_metrics(y_te, svm.predict(to_svm(X_te_svm)))
        print_metrics(norm_results["SVM"], "SVM")
    else:
        print(f"\n  ── [SVM] skipped for {norm_name} (no catch22 features) ──")
        norm_results["SVM"] = None

    # ── [B] ROCKET ────────────────────────────────────────────
    rocket = RocketClassifier(num_kernels=10_000, random_state=42, n_jobs=-1)
    rocket.fit(to_rocket(X_tr_raw), y_tr)
    norm_results["ROCKET"] = compute_metrics(y_te, rocket.predict(to_rocket(X_te_raw)))
    print_metrics(norm_results["ROCKET"], "ROCKET")

    # ── [C] GRU ───────────────────────────────────────────────
    y_tr_cat = to_categorical(y_tr, num_classes=n_classes)
    y_te_cat = to_categorical(y_te, num_classes=n_classes)

    tf.random.set_seed(42)
    gru_model = Sequential([
        GRU(64, input_shape=(n_timepoints, n_channels), return_sequences=True),
        Dropout(0.3),
        GRU(32, return_sequences=False),
        Dropout(0.3),
        Dense(32, activation="relu"),
        Dense(n_classes, activation="softmax")
    ], name=f"GRU_{norm_name}")

    gru_model.compile(optimizer="adam",
                      loss="categorical_crossentropy",
                      metrics=["accuracy"])

    gru_model.fit(
        to_gru(X_tr_raw), y_tr_cat,
        epochs=30,
        batch_size=32,
        class_weight=CLASS_WEIGHTS,
        validation_split=0.1,
        verbose=0
    )

    gru_pred = np.argmax(gru_model.predict(to_gru(X_te_raw), verbose=0), axis=1)
    norm_results["GRU"] = compute_metrics(y_te, gru_pred)
    print_metrics(norm_results["GRU"], "GRU")

    results[norm_name] = norm_results

# ══════════════════════════════════════════════════════════════
# SUMMARY TABLE
# ══════════════════════════════════════════════════════════════
rows = []
for norm_name, classifiers in results.items():
    for clf_name, metrics in classifiers.items():
        if metrics is None:
            continue
        rows.append({
            "Normalization":  norm_name,
            "Classifier":     clf_name,
            "TSS":            metrics["tss"],
            "Class 1 Recall": metrics["class1_recall"],
            "Class 1 F1":     metrics["class1_f1"],
            "Macro F1":       metrics["macro_f1"],
            "Accuracy":       metrics["accuracy"],
        })

df_results = (pd.DataFrame(rows)
                .set_index(["Normalization", "Classifier"])
                .sort_values("TSS", ascending=False))

print("\n\n" + "═"*70)
print("  FINAL COMPARISON  (sorted by TSS — primary metric)")
print("═"*70)
print(df_results.to_string())

best_norm_by_tss = (df_results.reset_index()
                              .groupby("Normalization")["TSS"]
                              .mean()
                              .idxmax())
best_single_row = df_results["TSS"].idxmax()

print(f"\n  ✓ Best single result:  {best_single_row[0]} / {best_single_row[1]}"
      f"  →  TSS = {df_results.loc[best_single_row, 'TSS']}")
print(f"  ✓ Best normalization overall (avg TSS): {best_norm_by_tss}")

# ══════════════════════════════════════════════════════════════
# NOTEBOOK 3 HANDOFF
# ══════════════════════════════════════════════════════════════
best_norm_dataset = datasets[best_norm_by_tss]

with open("./best_norm_for_nb3.pkl", "wb") as f:
    pickle.dump({
        "X_train": best_norm_dataset["X_train"],
        "y_train": best_norm_dataset["y_train"],
        "X_test":  best_norm_dataset["X_test"],
        "y_test":  best_norm_dataset["y_test"],
    }, f)

print(f"\n  ✓ Saved to ./best_norm_for_nb3.pkl")
print(f"  ✓ Notebook 3 will use: '{best_norm_by_tss}' normalization")
print(f"  ✓ Keys: X_train {best_norm_dataset['X_train'].shape}, "
      f"X_test {best_norm_dataset['X_test'].shape}")

✅  Raw datasets loaded:
    per_file    train (12455, 288, 10)  test (3559, 288, 10)
    per_row     train (12455, 288, 10)  test (3559, 288, 10)

✅  Catch22 datasets loaded:
    per_file    train (12455, 194)  test (3559, 194)
    per_row     train (12455, 195)  test (3559, 195)

✅  Key alignment confirmed: ['per_file', 'per_row']

════════════════════════════════════════════════════════════
  NORMALIZATION: PER_FILE
════════════════════════════════════════════════════════════

  ── [SVM] ──────────────────────────────────────
  TSS:            0.5114  ← primary metric
  Macro F1:       0.5406
  Class 1 F1:     0.1231
  Class 1 Recall: 0.5882
  Class 1 Prec.:  0.0687
  Accuracy:       0.9199  (misleading at 104:1)
  Confusion Matrix → TP:20  FP:271  TN:3254  FN:14

  ── [ROCKET] ──────────────────────────────────────
  TSS:            0.0871  ← primary metric
  Macro F1:       0.5707
  Class 1 F1:     0.1463
  Class 1 Recall: 0.0882
  Class 1 Prec.:  0.4286
  Accuracy:       0.9902  (

### 